# Spark sources and sinks
## Spark data sources
### External data source
- Lets assume that we have a batch processing requirement 
- Use a data integration tool to bring data to the distributed storage and then we start processing it.
- The reason for preferring this two step approach : 
    - Modularity
        - Bringing data correctly and effitiently to your lake is a complex goal in itself. The idea here is to decouple the ingestion from the processing to improve the managability.
    - Load Balance
        - The source systems would have been designed for a specific purpose and the capacity of the source systems would have been planned accordingly
    - Security
        - Now if you want to connect your spark work load to these systems then you must have to re-plan your source system's capacity and those security aspects of those systems.
    - Flexibility
        - We wnat to use the right tool for the right purpose.
        - Spark was designed for data processing its not suited for data ingestion.
        - Hence even though spark provide a way to connect to an external source usually people avoid to do so.
### Internal data source
- Internal data source could be HDFS or some cloud storage
- The mechanism of reading data is the same in case of HDFS and cloud storage. The only difference is in the file format.

### NOTE : 
- Working with the data source is all about reading the data. 
- Working with the sink is all about writing the data.


## Spark DataFrame reader API
### How to use dataFrame reader for csv, json and parquet data sources
- When we import data from csv or json file format and use inferschema to true in case of csv then the datatypes of the columns are mostly correct except for the date type columns. The datatype of the date type columns for some reason is always set to string 
- In case of importing data from a parquet file we don't have to worry about the dataType of the columns because it contains the shema information already included in the data file. So in this case I don't have to specify the schema explicitly. However the data-file must contain the correct schema.
- Because of this reason it is recommended to use parquet file format for as long as possible.
### Explicitly set the schema for your dataFrames
- DataFrame schema is all about setting the column name and its appropriate dataTypes, However you should also know the spark supported dataTypes.
- Spark DataTypes : Apache spark comes with its own dataTypes when it comes to defining the spark dataFrame schema.
    - IntegerType
    - LongType
    - FloatType
    - DoubleType
    - StringType
    - DateType
    - TimestampType
    - ArrayType
    - MapType
- Why does spark maintains its own datatypes instead of using the language specific types? for example python has its own datatypes why don't we use python dataTypes to define the dataFrame schema?
    - Spark is like a compiler it compiles the high level api code into lower level RDD operations.
    - During this compilation process it generates different execution plans and also perform a bunch of optimizations. All this is not possible without maintaining its own type information.
- Spark allows you to define the spark dataFrame schema in two ways 
    - Setting up spark DataFrame schema Programmatically
        - schema.py
        ```python
        from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType, TimestampType

        """
        This class defines the spark dataFrame schema using Programmatical method
        """
        class FlightSchemaMixin:
            def return_flight_df_schema(self):
                flight_schema = StructType([
                    StructField("FL_DATE", DateType(), True),
                    StructField("OP_CARRIER", StringType(), True),
                    StructField("OP_CARRIER_FL_NUM", IntegerType(), True),
                    StructField("ORIGIN", StringType(), True),
                    StructField("ORIGIN_CITY_NAME", StringType(), True),
                    StructField("DEST", StringType(), True),
                    StructField("DEST_CITY_NAME", StringType(), True),
                    StructField("CRS_DEP_TIME", IntegerType(), True),
                    StructField("DEP_TIME", IntegerType(), True),
                    StructField("WHEELS_ON", IntegerType(), True),
                    StructField("TAXI_IN", IntegerType(), True),
                    StructField("CRS_ARR_TIME", IntegerType(), True),
                    StructField("ARR_TIME", IntegerType(), True),
                    StructField("CANCELLED", IntegerType(), True),
                    StructField("DISTANCE", IntegerType(), True)
                ])
                return flight_schema
        ```
        - ingest.py
        ```python
        # --- Add project root to sys.path ---
        import os 
        import sys
        CURRENT_DIR = os.path.dirname(os.path.abspath(__file__))
        PROJECT_ROOT = os.path.dirname(CURRENT_DIR)
        if PROJECT_ROOT not in sys.path:
            sys.path.insert(0, PROJECT_ROOT)

        from .logger import Log4j
        from .app_monitor import GetDataFrameMemory

        from spark_dataFrame_schema.spark_dataframe_schema import FlightSchemaMixin

        """
        This class ingest data from csv, json and parquet file format
        """
        class IngestData():
            def __init__(self,spark):
                self.spark_object = spark
                self.logger = Log4j(spark)
                self.metrics = GetDataFrameMemory(spark)
                self.df_schema = FlightSchemaMixin()

            def import_data_csv(self,file_dir):
                try:
                    spark_df = (
                            self.spark_object
                            .read
                            .format("csv")
                            .option("header","true")
                            # .option("inferschema","true")
                            .schema(self.df_schema.return_flight_df_schema())
                            # Set the mode for error if the schema don't match
                            .option("mode","FAILFAST")
                            # Set the date string format
                            .option("dateFormat","M/d/y")
                            .load(file_dir)
                    )
                    self.log_df_metrics(spark_df=spark_df,file_dir=file_dir)
                    return spark_df
                except Exception as e:
                    self.logger.error(str(e))

            # utility methods
            def log_df_metrics(self,spark_df,file_dir):
                self.logger.info(f"import_data_csv :: spark_df created successfully from {file_dir} dataset file")
                self.logger.info(f"import_data_csv :: The memory taken by the spark dataFrame is = {self.metrics.get_mem_usage(spark_df).get("mem")} MB")
                schema_str = spark_df._jdf.schema().treeString()
                self.logger.debug(f"Spark DataFrame Schema (expanded): {schema_str}")
        ```
        - main.py
        ```python
        from pyspark.sql import SparkSession
        # import related to logging
        from lib.logger import Log4j, LogSparkDataframe
        # import related to custom spark configurations
        from lib.utils import get_spark_app_config
        # logging related imports 
        import os

        # Imports related to ingest data
        from lib.ingest_data import IngestData
        # Transform data
        from transformations.dataframe_transformations import DataFrameTransformations

        if __name__ == "__main__":
            # logging related logic
            # Get the current project's directory
            project_dir = os.path.dirname(os.path.abspath(__file__))
            # Get the Log4j.properties file directory
            log4j_config_path = os.path.join(project_dir, "log4j_properties", "log4j.properties")
            # Save the directory where the generated log files must reside
            log_dir = os.path.join(project_dir, "log4j_properties", "logs")
            # Create the directory where the log files must be kept if not present
            os.makedirs(log_dir, exist_ok=True)

            conf = get_spark_app_config()
            spark = (
                SparkSession
                .builder
                .config(conf=conf)
                .config("spark.driver.extraJavaOptions",
                        f"-Dlog4j.configuration=file:{log4j_config_path} -Dcustom.log.dir={log_dir}")
                .config("spark.executor.extraJavaOptions",
                        f"-Dlog4j.configuration=file:{log4j_config_path} -Dcustom.log.dir={log_dir}")
                .getOrCreate()
            )

            # initialize logger class 
            logger = Log4j(spark)

            # initialize the spark dataframe logger 
            sp_df_logger = LogSparkDataframe(spark)

            # logging some debug related stuff 
            logger.debug(f"log4j.properties file dir = {log4j_config_path}")
            logger.debug(f"log files dir = {log_dir}")
            logger.debug(f"log dir exists = {os.path.exists(log_dir)}")
            
            logger.info("Reading the data from the directory")
            dataset_dir = os.path.join(project_dir,"dataset")
            # file_name = "sf-fire-calls.csv" nor mally we provide the file name by hard coding it in the app 
            # But here the dataset file name is supplied via spark.conf file
            file_name = conf.get("file_name_csv")
            file_dir = os.path.join(dataset_dir,file_name)
            logger.debug(f"file_name_csv dir = {file_dir}")
            
            # The function must taken in file_dir csv file and then returns a spark dataFrame
            # import data from a csv file
            ingest_data = IngestData(spark)
            spark_df = ingest_data.import_data_csv(file_dir=file_dir)

            # log spark dataframe
            sp_df_logger.log_df(spark_df=spark_df,spark_df_name="spark_df")

            # import data from a json file
            file_name = conf.get("file_name_json")
            file_dir = os.path.join(dataset_dir,file_name)
            logger.debug(f"file_name_json dir = {file_dir}")
            spark_df_json = ingest_data.import_data_json(file_dir=file_dir)

            # log spark dataframe
            sp_df_logger.log_df(spark_df=spark_df,spark_df_name="spark_df_json")

            # import data from a parquet file
            file_name = conf.get("file_name_parquet")
            file_dir = os.path.join(dataset_dir,file_name)
            logger.debug(f"file_name_json dir = {file_dir}")
            spark_df_parquet = ingest_data.import_data_parquet(file_dir=file_dir)

            # log spark dataframe
            sp_df_logger.log_df(spark_df=spark_df,spark_df_name="spark_df_parquet")

            # This line is for debugging only comment after <required to see the partitions of spark dataFrame>
            # input("Please enter")
            spark.stop()
        ```
    - Setting up spark DataFrame using DDL String
        - schema.py
        ```python
        from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType, TimestampType


        class FlightSchemaMixin:

            """
            This method defines the spark dataFrame schema using DDL method
            """
            def return_flight_schema_ddl(self):
                flight_schema_ddl = """
                    FL_DATE DATE,
                    OP_CARRIER STRING,
                    OP_CARRIER_FL_NUM INT,
                    ORIGIN STRING,
                    ORIGIN_CITY_NAME STRING,
                    DEST STRING,
                    DEST_CITY_NAME STRING,
                    CRS_DEP_TIME INT,
                    DEP_TIME INT,
                    WHEELS_ON INT,
                    TAXI_IN INT,
                    CRS_ARR_TIME INT,
                    ARR_TIME INT,
                    CANCELLED INT,
                    DISTANCE INT
                """
                return flight_schema_ddl
        ```
        - ingest_data.py
        ```python
        # --- Add project root to sys.path ---
        import os 
        import sys
        CURRENT_DIR = os.path.dirname(os.path.abspath(__file__))
        PROJECT_ROOT = os.path.dirname(CURRENT_DIR)
        if PROJECT_ROOT not in sys.path:
            sys.path.insert(0, PROJECT_ROOT)

        from .logger import Log4j
        from .app_monitor import GetDataFrameMemory

        from spark_dataFrame_schema.spark_dataframe_schema import FlightSchemaMixin

        """
        This class ingest data from csv, json and parquet file format
        """
        class IngestData():
            def __init__(self,spark):
                self.spark_object = spark
                self.logger = Log4j(spark)
                self.metrics = GetDataFrameMemory(spark)
                self.df_schema = FlightSchemaMixin()

            def import_data_json(self,file_dir):
                try:
                    spark_df = (
                        self.spark_object
                        .read
                        .format("json")
                        .schema(self.df_schema.return_flight_schema_ddl())
                        .option("dateFormat","M/d/y")
                        .load(file_dir)
                    )
                    self.log_df_metrics(spark_df=spark_df,file_dir=file_dir)
                    return spark_df
                except Exception as e:
                    self.logger.error(str(e))

            def import_data_parquet(self,file_dir):
                try:
                    spark_df = (
                        self.spark_object
                        .read
                        .format("parquet")
                        .load(file_dir)
                    )
                    self.log_df_metrics(spark_df=spark_df,file_dir=file_dir)
                    return spark_df
                except Exception as e:
                    self.logger.error(str(e))

            # utility methods
            def log_df_metrics(self,spark_df,file_dir):
                self.logger.info(f"import_data_csv :: spark_df created successfully from {file_dir} dataset file")
                self.logger.info(f"import_data_csv :: The memory taken by the spark dataFrame is = {self.metrics.get_mem_usage(spark_df).get("mem")} MB")
                schema_str = spark_df._jdf.schema().treeString()
                self.logger.debug(f"Spark DataFrame Schema (expanded): {schema_str}")
         ```
         - main.py
         ```python
         from pyspark.sql import SparkSession
        # import related to logging
        from lib.logger import Log4j, LogSparkDataframe
        # import related to custom spark configurations
        from lib.utils import get_spark_app_config
        # imports related to exporting dataframe
        from lib.write_df import ExportSparkDataFrame
        # logging related imports 
        import os

        # Imports related to ingest data
        from lib.ingest_data import IngestData
        # Transform data
        from transformations.dataframe_transformations import DataFrameTransformations

        if __name__ == "__main__":
            # logging related logic
            # Get the current project's directory
            project_dir = os.path.dirname(os.path.abspath(__file__))
            # Get the Log4j.properties file directory
            log4j_config_path = os.path.join(project_dir, "log4j_properties", "log4j.properties")
            # Save the directory where the generated log files must reside
            log_dir = os.path.join(project_dir, "log4j_properties", "logs")
            # Create the directory where the log files must be kept if not present
            os.makedirs(log_dir, exist_ok=True)

            conf = get_spark_app_config()
            spark = (
                SparkSession
                .builder
                .config(conf=conf)
                .config("spark.driver.extraJavaOptions",
                        f"-Dlog4j.configuration=file:{log4j_config_path} -Dcustom.log.dir={log_dir}")
                .config("spark.executor.extraJavaOptions",
                        f"-Dlog4j.configuration=file:{log4j_config_path} -Dcustom.log.dir={log_dir}")
                .getOrCreate()
            )

            # initialize logger class 
            logger = Log4j(spark)

            # initialize the spark dataframe logger 
            sp_df_logger = LogSparkDataframe(spark)

            # logging some debug related stuff 
            logger.debug(f"log4j.properties file dir = {log4j_config_path}")
            logger.debug(f"log files dir = {log_dir}")
            logger.debug(f"log dir exists = {os.path.exists(log_dir)}")
            
            logger.info("Reading the data from the directory")
            dataset_dir = os.path.join(project_dir,"dataset")

            # INGETING DATA FROM VARIOUS FILE FORMATS STARTS
            # file_name = "sf-fire-calls.csv" nor mally we provide the file name by hard coding it in the app 
            # But here the dataset file name is supplied via spark.conf file
            file_name = conf.get("file_name_csv")
            file_dir = os.path.join(dataset_dir,file_name)
            logger.debug(f"file_name_csv dir = {file_dir}")
            
            # The function must taken in file_dir csv file and then returns a spark dataFrame
            # import data from a csv file
            ingest_data = IngestData(spark)
            spark_df = ingest_data.import_data_csv(file_dir=file_dir)

            # log spark dataframe
            sp_df_logger.log_df(spark_df=spark_df,spark_df_name="spark_df")

            # import data from a json file
            file_name = conf.get("file_name_json")
            file_dir = os.path.join(dataset_dir,file_name)
            logger.debug(f"file_name_json dir = {file_dir}")
            spark_df_json = ingest_data.import_data_json(file_dir=file_dir)

            # log spark dataframe
            sp_df_logger.log_df(spark_df=spark_df,spark_df_name="spark_df_json")

            # import data from a parquet file
            file_name = conf.get("file_name_parquet")
            file_dir = os.path.join(dataset_dir,file_name)
            logger.debug(f"file_name_json dir = {file_dir}")
            spark_df_parquet = ingest_data.import_data_parquet(file_dir=file_dir)

            # log spark dataframe
            sp_df_logger.log_df(spark_df=spark_df,spark_df_name="spark_df_parquet")
            # INGETING DATA FROM VARIOUS FILE FORMATS ENDS

            # This line is for debugging only comment after <required to see the partitions of spark dataFrame>
            # input("Please enter")
            spark.stop()
         ```
- Spark dataframe schema is of StructType which is made up of a list of StructField
    - The StructField takes in two argument one is the column name and the second one is the dataType.
    - Spark must throw an error if the dataFrame schema doesn't match with the incoming data from the csv or json file however we will have to setup the mode for getting the error or else spark will sielently fail to parse the column whose datatype don't match with the defined spark schema.
### Export spark dataFrame in paraquet, avro and json format files
- There a some key things to remeber before we move forward 
    - When exporting dataFrames you have to make sure that the dataFrame is partition in such a way that the partition file size in your file system ranges from 500MB to 1GB not too small and not too big
- Partitioning our data will give us two benefits 
    - parallel processing
        - In this case you will have two divide the no of rows in such a way that each partition has same number of rows
    - partition elimination for certain kind of read operations
        - Here in this case you will have to partition the data based on specific columns using the partition by method
        ```python
        def export_df_json(self,save_mode : str = "overwrite", output_path : str = "",column_list : list = []):
        try:
            (
                self.spark_df.write
                .format("json")
                .mode(save_mode)
                .option("path",output_path)
                .partitionBy(column_list[0],column_list[1])
                .save()
            )
        except Exception as e:
            self.logger.error(str(e))
            raise
        ```
- FINAL CODE
- main.py
```python
from pyspark.sql import SparkSession
# import related to logging
from lib.logger import Log4j, LogSparkDataframe
# import related to custom spark configurations
from lib.utils import get_spark_app_config
# imports related to exporting dataframe
from lib.write_df import ExportSparkDataFrame
# logging related imports 
import os

# Imports related to ingest data
from lib.ingest_data import IngestData
# Transform data
from transformations.dataframe_transformations import DataFrameTransformations

if __name__ == "__main__":
    # logging related logic
    # Get the current project's directory
    project_dir = os.path.dirname(os.path.abspath(__file__))
    # Get the Log4j.properties file directory
    log4j_config_path = os.path.join(project_dir, "log4j_properties", "log4j.properties")
    # Save the directory where the generated log files must reside
    log_dir = os.path.join(project_dir, "log4j_properties", "logs")
    # Create the directory where the log files must be kept if not present
    os.makedirs(log_dir, exist_ok=True)

    conf = get_spark_app_config()
    spark = (
        SparkSession
        .builder
        .config(conf=conf)
        .config("spark.driver.extraJavaOptions",
                f"-Dlog4j.configuration=file:{log4j_config_path} -Dcustom.log.dir={log_dir}")
        .config("spark.executor.extraJavaOptions",
                f"-Dlog4j.configuration=file:{log4j_config_path} -Dcustom.log.dir={log_dir}")
        .config("spark.jars.packages", "org.apache.spark:spark-avro_2.13:4.0.1")
        .getOrCreate()
    )

    # initialize logger class 
    logger = Log4j(spark)

    # initialize the spark dataframe logger 
    sp_df_logger = LogSparkDataframe(spark)

    # logging some debug related stuff 
    logger.debug(f"log4j.properties file dir = {log4j_config_path}")
    logger.debug(f"log files dir = {log_dir}")
    logger.debug(f"log dir exists = {os.path.exists(log_dir)}")
    
    logger.info("Reading the data from the directory")
    dataset_dir = os.path.join(project_dir,"dataset")

    # INGETING DATA FROM VARIOUS FILE FORMATS STARTS
    # file_name = "sf-fire-calls.csv" nor mally we provide the file name by hard coding it in the app 
    # But here the dataset file name is supplied via spark.conf file
    file_name = conf.get("file_name_csv")
    file_dir = os.path.join(dataset_dir,file_name)
    logger.debug(f"file_name_csv dir = {file_dir}")
    
    # The function must taken in file_dir csv file and then returns a spark dataFrame
    # import data from a csv file
    ingest_data = IngestData(spark)
    spark_df = ingest_data.import_data_csv(file_dir=file_dir)

    # log spark dataframe
    sp_df_logger.log_df(spark_df=spark_df,spark_df_name="spark_df")

    # import data from a json file
    file_name = conf.get("file_name_json")
    file_dir = os.path.join(dataset_dir,file_name)
    logger.debug(f"file_name_json dir = {file_dir}")
    spark_df_json = ingest_data.import_data_json(file_dir=file_dir)

    # log spark dataframe
    sp_df_logger.log_df(spark_df=spark_df,spark_df_name="spark_df_json")

    # import data from a parquet file
    file_name = conf.get("file_name_parquet")
    file_dir = os.path.join(dataset_dir,file_name)
    logger.debug(f"file_name_json dir = {file_dir}")
    spark_df_parquet = ingest_data.import_data_parquet(file_dir=file_dir)

    # log spark dataframe
    sp_df_logger.log_df(spark_df=spark_df,spark_df_name="spark_df_parquet")
    # INGETING DATA FROM VARIOUS FILE FORMATS ENDS

    # EXPORTING DATAFRAME STARTS
    # export dataframe in paraquet format
    project_dir
    export_dir = os.path.join(project_dir, "export_df", "flight_data_paraquet")
    # Create the directory where the exported dataframe files must be kept if not present
    os.makedirs(export_dir, exist_ok=True)
    # initializing dtaframe exporter class 
    export_obj = ExportSparkDataFrame(spark_df,spark)
    export_obj.export_df_parquet(save_mode="overwrite",output_path=export_dir,max_rec=1000)

    # export data in avro format
    project_dir
    export_dir = os.path.join(project_dir, "export_df", "flight_data_avro")
    # Create the directory where the exported dataframe files must be kept if not present
    os.makedirs(export_dir, exist_ok=True)
    # initializing dtaframe exporter class 
    export_obj = ExportSparkDataFrame(spark_df,spark)
    export_obj.export_df_avro(save_mode="overwrite",output_path=export_dir)

    # export data in JSON format
    project_dir
    export_dir = os.path.join(project_dir, "export_df", "flight_data_json")
    # Create the directory where the exported dataframe files must be kept if not present
    os.makedirs(export_dir, exist_ok=True)
    # initializing dtaframe exporter class 
    export_obj = ExportSparkDataFrame(spark_df,spark)
    export_obj.export_df_json(save_mode="overwrite",output_path=export_dir,column_list=["OP_CARRIER","ORIGIN"])
    # EXPORTING DATAFRAME ENDS

    # This line is for debugging only comment after <required to see the partitions of spark dataFrame>
    # input("Please enter")
    spark.stop()
```
- write_df.py
```python
from lib.logger import Log4j
class ExportSparkDataFrame:
    def __init__(self,spark_df,spark):
        self.spark_df = spark_df
        self.logger = Log4j(spark)
    def export_df_parquet(self,save_mode: str = "overwrite",output_path : str="",max_rec : int = 100):
        try:
            (
                self.spark_df.write
                .format("parquet")
                .mode(save_mode)
                .option("path",output_path)
                .option("maxRecordsPerFile",max_rec)
                .save()
            )
            return True
        except Exception as e:
            self.logger.error(str(e))
            raise
    
    def export_df_avro(self,save_mode: str = "overwrite", output_path: str = ""):
        try:
            (
                self.spark_df.write
                .format("avro")
                .mode(save_mode)
                .option("path",output_path)
                .save()
            )
            return True
        except Exception as e:
            self.logger.error(str(e))
            raise

    def export_df_json(self,save_mode : str = "overwrite", output_path : str = "",column_list : list = []):
        try:
            (
                self.spark_df.write
                .format("json")
                .mode(save_mode)
                .option("path",output_path)
                .partitionBy(column_list[0],column_list[1])
                .save()
            )
        except Exception as e:
            self.logger.error(str(e))
            raise
```
#### Explaiantion : 
- ```.option("maxRecordsPerFile",max_rec)```
    - The .option("maxRecordsPerFile", max_rec) configuration tells Spark how many rows (records) it should write into each output file when saving a DataFrame.
    - When Spark writes data (for example, to Parquet, JSON, CSV, or Avro), it normally creates one file per partition of your DataFrame.
        - If I have 200 partitions → Spark will produce ~200 output files.
        - File sizes can be very uneven — depending on how my data is distributed.
        - By setting: I am saying: No matter how many records are in a partition, don’t put more than 1,000,000 rows in a single file — split them automatically if needed.
- ```.partitionBy(column_list[0], column_list[1])```
    - When you write a Spark DataFrame to disk (in formats like Parquet, Avro, JSON, or CSV), the .partitionBy() method tells Spark to physically organize the output files into subdirectories, based on the unique values of the columns you specify.
    - Spark looks at all unique combinations of the columns I pass to .partitionBy().
    - For each unique combination, it writes one or more files under a directory named after the column values.
    - These subdirectories are automatically created and encoded as col=value pairs.
    - **Why this is useful?**
        - When I later read the data:
        ```python
        df = spark.read.json("/data/output")
        df.filter("year = 2024 AND month = '01'")
        ```
        - Spark will only read files from year=2024/month=01/, skipping all others.
        → This is called partition pruning — it can drastically speed up queries.
        - Organized data lake structure:
            - Makes it easy to browse or manage data by time period, region, or category.
        - Scales well for large datasets:
            - Commonly used with Hive, Delta Lake, or S3-based data lakes.
    - **Common pitfalls :**
        - Don’t partition on high-cardinality columns (e.g., user_id, transaction_id). This can create millions of tiny directories — very inefficient.
        - Column must exist in DataFrame 
            - If column_list includes a non-existent column, Spark will throw an error:
            ```bash
            AnalysisException: Partition column `foo` not found in schema
            ```
            - Partition columns not part of file content (for some formats)
            - In Parquet/Avro, partition columns are usually not stored inside the file, but inferred from the directory structure.
            - In JSON/CSV, the columns are written into the file, too.
    - **Best practices :**
        -  Partition by low- to medium-cardinality columns (e.g., year, month, country).
        - Combine with .option("maxRecordsPerFile", ...) to control file sizes.
        - Use repartition() before writing if partitions are unbalanced.
        ```python
        (
            spark_df
            .repartition("year", "month")
            .write
            .format("json")
            .mode("overwrite")
            .option("path", output_path)
            .option("maxRecordsPerFile", 500000)
            .partitionBy("year", "month")
            .save()
        )
        ```
#### Errors I faced 
```bash
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/home/aditya/github/Deep-learning-prerequisite/pyspark/Spark_sources_and_sinks/SparkSchemaDemo/main_sch_app.py", line 37, in <module>
    .getOrCreate()
     ~~~~~~~~~~~^^
  File "/home/aditya/miniconda3/envs/pyspark/lib/python3.13/site-packages/pyspark/sql/session.py", line 556, in getOrCreate
    sc = SparkContext.getOrCreate(sparkConf)
  File "/home/aditya/miniconda3/envs/pyspark/lib/python3.13/site-packages/pyspark/core/context.py", line 523, in getOrCreate
    SparkContext(conf=conf or SparkConf())
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/aditya/miniconda3/envs/pyspark/lib/python3.13/site-packages/pyspark/core/context.py", line 205, in __init__
    SparkContext._ensure_initialized(self, gateway=gateway, conf=conf)
    ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/aditya/miniconda3/envs/pyspark/lib/python3.13/site-packages/pyspark/core/context.py", line 444, in _ensure_initialized
    SparkContext._gateway = gateway or launch_gateway(conf)
                                       ~~~~~~~~~~~~~~^^^^^^
  File "/home/aditya/miniconda3/envs/pyspark/lib/python3.13/site-packages/pyspark/java_gateway.py", line 111, in launch_gateway
    raise PySparkRuntimeError(
    ...<2 lines>...
    )
pyspark.errors.exceptions.base.PySparkRuntimeError: [JAVA_GATEWAY_EXITED] Java gateway process exited before sending its port number.
``` 
- This error occurs due to mismatch of scala version. Remember the same pyspark version can be compiled wih different scala version.
- In my case I added this config 
```python
.config("spark.jars.packages", "org.apache.spark:spark-avro_2.12:4.0.1")
```
- After getting this error I confirmed which version of scala is being used under the hood of my pyspark installation by typing 
```bash
conda activate pyspark --> activate your env
pyspark --version
```
You should get output like this 
```bash
WARNING: Using incubator modules: jdk.incubator.vector
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/11/02 17:44:32 WARN Utils: Your hostname, aditya-IdeaPad-5-15ITL05, resolves to a loopback address: 127.0.1.1; using 192.168.1.19 instead (on interface wlp0s20f3)
25/11/02 17:44:32 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Welcome to
      ____              __
     / __/__  ___ _____/ /__
    _\ \/ _ \/ _ `/ __/  '_/
   /___/ .__/\_,_/_/ /_/\_\   version 4.0.1
      /_/
                        
Using Scala version 2.13.16, OpenJDK 64-Bit Server VM, 17.0.15
Branch HEAD
Compiled by user runner on 2025-09-02T03:10:51Z
Revision 29434ea766b0fc3c3bf6eaadb43a8f931133649e
Url https://github.com/apache/spark
Type --help for more information.

```
- As you can see there is a clear mismatch in the scala version if you compare these two ```Using Scala version 2.13.16, OpenJDK 64-Bit Server VM, 17.0.15``` vs ```.config("spark.jars.packages", "org.apache.spark:spark-avro_2.12:4.0.1")```
#### Solution
- Update the import when declaring dependecis during Spark session building process ```.config("spark.jars.packages", "org.apache.spark:spark-avro_2.13:4.0.1")```

## Spark Databases and Tables
![spark_data_and_table](images/spark_data_and_table.png)
- Aapche spark is not only a set of apis and a data processing engine, it is a database in itself.
- So you can create a database the spark. 
- Once you have a database you can create database tables and views.
- These tables and views reside inside your database
- The table has got two parts 
    - Table data
        - It resides as datafiles in your distributed storage by default it is a parquet file.
    - Table meta data
        - The metadata is stored in the meta store called catalog.
        - The meta store holds the information about the table and its data such as (schema, tablename, database name, column names, partitions, the physical location where the actual data resides). all of this is stored in a central meta store called catalog.
        - By default spark comes with an in-memory catalog which is maintained per spark session. This information goes away when the session ends.
        - So we needed a persistant and durable meta store. 
        - Spark decided to re-use the apache HIVe meta store.
    - Spark allows you to create two types of tables:
        - Managed tables
            - For managed tables spark manages both the meta-data and the data.
            - If cretae a managed tables then spark is going to create two things.
                - Create and store meta-data about the table in the meta store.
                - Then the spark is going to write the data inside a pre-defined directory location.
                - This directory is known as spark sql warehouse directory
                - The spark sql warehouse directory is the base location where all your managed tables are stored and your cluster admin is going to set this base directory location for you. 
                - NOTE : You cannot and should not change the base directory for the spark sql warehouse at runtime 
                - We prefer using managed tables because they prefer some additional features, such as bucketing and sorting.
                - All the future improvements in the spark sql will also target managed tables.
                - Un-managed tables are external tables and spark do not have any control over them.
                - They are designed for re-using the existing data in spark sql and it should be used in those senarios only
        - Unmanaged tables (External tables)
            - They are the same with respect to the meta-data
            - However they are different in terms of data storage location. 
            - When you create an un-managed table spark is going to do only one thing 
                - It will create the meta-data for your table and store it in the meta-store.
            - In this case when you create an unmanaged table then you must specify the data directory location for your table
            - This facility is not provided in case of managed table because the managed table must be stored inside the warehouse directory.
            - However the unmanaged tables are to give you the flexibility to store your data at your prefered location.
            - Think of it like this :
                - You already have some data is is stored at some directory location and you want to use spark sql satement  on this dataset. 
                - However spark sl engine doesn't know anything about this data.
                - So you can create an un-managed table and map the same data to a spark table.
                - Now spark will create meta-data and store it.
                - This will you to run your spark sql statements on this data.
                - As a side-effect if you drop your managed table spark is going to delete the meta-data and the data as well.
                - If you drop the un-managed table then spark is going to remove the meta-data and it will not touch the data files.
                    - This happens because un-managed tables are designed to temorarily map the existing data and use it in the spark sql.
                    - Once you are doen using it drop the un-managed table and only the meta-data is deleted and the original files are not touched.

### Create a managed table and acceess the catalog
#### Create a managed table in apache spark and save data form a spark dataframe inside that table.
#### final code :
- cleanup.py : This file prevents the main script from failing in case where the main_app or main.py file is re-run
- you will find the code related to this in my github : https://github.com/aryan68125/Deep-learning-prerequisite/tree/master/pyspark/Spark_sources_and_sinks/SparkSqlTables
```python
import os
import shutil

import time

"""
This class will cleanup the data when the spark application re-runs

The files like logs, metastore , spark-warehouse etc.. will be cleaned up (deleted from the file system)
"""
class CleanupAppFileSystemOnReRun:
    def __init__(self,project_dir):
        self.project_dir = project_dir

    def execute_cleanup(self,clean_logs:bool = False):
        self.derby_logs_cleanup()
        self.spark_warehouse_cleanup()
        self.metastore_cleanup()
        if clean_logs == True:
            self.logs_cleanup()
        # time.sleep(5)

    """This will cleanup the derby.logs"""
    def derby_logs_cleanup(self):
        try:
            derby_logs_dir = os.path.join(self.project_dir, "derby.log")
            if os.path.exists(derby_logs_dir):
                os.remove(derby_logs_dir)
                print(f"Deleted existing derby.log file: {derby_logs_dir}")
            else:
                print(f"derby.log file does not exists: {derby_logs_dir}")
        except Exception as e:
            print(str(e))
            raise
    
    """This will cleanup the spark_warehouse"""
    def spark_warehouse_cleanup(self):
        try:
            spark_warehouse_dir = os.path.join(self.project_dir, "spark-warehouse")
            if os.path.exists(spark_warehouse_dir):
                shutil.rmtree(spark_warehouse_dir)
                print(f"Deleted existing spark-warehouse directory: {spark_warehouse_dir}")
            else:
                print(f"spark-warehouse directory does not exists: {spark_warehouse_dir}")
        except Exception as e:
            print(str(e))
            raise

    """This will cleanup the metastore_db"""
    def metastore_cleanup(self):
        try:
            metastore_dir = os.path.join(self.project_dir, "metastore_db")
            if os.path.exists(metastore_dir):
                shutil.rmtree(metastore_dir)
                print(f"Deleted existing metastore directory: {metastore_dir}")
            else:
                print(f"metastore directory does not exists: {metastore_dir}")
        except Exception as e:
            print(str(e))
            raise

    """This will clean the logs folder"""
    def logs_cleanup(self):
        try:
            log_dir = os.path.join(self.project_dir, "log4j_properties", "logs")
            # delete the log directory
            if os.path.exists(log_dir):
                shutil.rmtree(log_dir)
                print(f"Deleted existing log directory: {log_dir}")
            else:
                print(f"Log directory does not exist: {log_dir}")
        except Exception as e:
            print(str(e))
            raise
```
- spark.conf
```bash
[SPARK_APP_CONFIGS]
saprk.app.name = SparkSqlTableDemo
spark.master = local[3]

# Setting up the dataset file name that are used in this application
file_name_csv = flight-time.csv
file_name_json = flight-time.json
file_name_parquet = flight-time.parquet

# Added a shuffle sort partitions to control the no of partitions of the spark dataFrame in the spark applicaiton
spark.sql.shuffle.partitions = 2

# Tell spark to save the created table in this database
db_name = airline_db
flight_table_name = flight_data
```
- load_df_data_into_table.py
```python
from .logger import Log4j
"""
This class will load the data in a spark dataFrame into a table
"""
class LoadSparkDFIntoTable: 
    def __init__(self,spark):
        self.spark = spark
        self.logger = Log4j(spark)
    
    """This method will save the spark dataFrame into a spark managed table"""
    def save_df_to_spark_managed_table(self,spark_df,partition_config:list = [],mode:str = "overwrite",db_name:str = "",table_name:str=""):
        try:
            if not db_name == "":
                # To be on the safer side create the db just in case
                query = f"""
                CREATE DATABASE IF NOT EXISTS {db_name}
                """
                self.spark.sql(query)
                # tell spark to use the created database 
                self.spark.catalog.setCurrentDatabase(db_name)
                """
                5 is the partition no
                OP_CARRIER = column name 
                ORIGIN = column name
                """
                writer = (
                    spark_df
                    .write
                    .mode(mode)
                )
                writer = self.partition_handler(partition_config,writer)
                writer.saveAsTable(table_name)
                self.logger.info(self.spark.catalog.listTables(db_name))
            else:
                writer = (
                    spark_df
                    .write
                    .mode(mode)
                )
                writer = self.partition_handler(partition_config,writer)
                writer.saveAsTable(table_name)
                self.logger.info(self.spark.catalog.listTables("default"))
        except Exception as e:
            self.logger.error(str(e))
    
    # util methods 
    def partition_handler(self,partition_config,writer):
        try:
            # Handle None or empty list
            if not partition_config:
                partition_config = []
                return writer

            num_buckets = None
            partition_cols = []

            # Detect if first element is an integer (buckets)
            if partition_config and isinstance(partition_config[0], int):
                num_buckets = partition_config[0]
                partition_cols = partition_config[1:]
            else:
                partition_cols = partition_config

            # Apply bucketing (if defined)
            if num_buckets and partition_cols:
                self.logger.debug("Applying bucketBy on the table")
                writer = writer.bucketBy(num_buckets, *partition_cols).sortBy(*partition_cols)
            elif partition_cols:
                self.logger.debug("Applying partitionBy on the table")
                writer = writer.partitionBy(*partition_cols)
            return writer
        except Exception as e:
            self.logger.error(str(e))
            raise

    def generate_logs(self, conf):
        try:
            db_name = conf.get("db_name")
            table_name = conf.get("flight_table_name")

            # Switch to the correct database
            if db_name:
                self.spark.catalog.setCurrentDatabase(db_name)

            # Get the first 24 records
            query = f"SELECT * FROM {table_name} LIMIT 24"
            result_df = self.spark.sql(query)

            # Capture the DataFrame as a formatted string (no stdout print)
            result_str = result_df._jdf.showString(24, 0, False)
            self.logger.info(f"\nFirst 24 records from table '{table_name}':\n{result_str}")

            # Get count value properly
            count_df = self.spark.sql(f"SELECT COUNT(*) as total FROM {table_name}")
            count_value = count_df.collect()[0]['total']
            self.logger.info(f"The number of records in table '{table_name}' => {count_value}")

            # Log schema
            self.logger.debug(f"Schema of table '{db_name}.{table_name}':")
            for field in result_df.schema.fields:
                self.logger.debug(f"  {field.name}: {field.dataType.simpleString()}")

        except Exception as e:
            self.logger.error(str(e))
            raise
```
- main.py
```python
from pyspark.sql import SparkSession
# import related to logging
from lib.logger import Log4j, LogSparkDataframe
# import related to custom spark configurations
from lib.utils import get_spark_app_config
# imports related to exporting dataframe
from lib.write_df import ExportSparkDataFrame
# import writing sparkdf to tables related stuff
from lib.load_df_data_into_table import LoadSparkDFIntoTable
# logging related imports 
import os

# Imports related to ingest data
from lib.ingest_data import IngestData
# Transform data
from transformations.dataframe_transformations import DataFrameTransformations

# imports related to cleanup when the main_app.py is re-run
from lib.clean_up_file_system import CleanupAppFileSystemOnReRun

if __name__ == "__main__":
    # logging related logic
    # Get the current project's directory
    project_dir = os.path.dirname(os.path.abspath(__file__))
    # cleanup loggic on main_app.py re-run
    # initialize the cleanup class
    cleanup = CleanupAppFileSystemOnReRun(project_dir)
    cleanup.execute_cleanup(clean_logs=True)

    # Get the Log4j.properties file directory
    log4j_config_path = os.path.join(project_dir, "log4j_properties", "log4j.properties")
    # Save the directory where the generated log files must reside
    log_dir = os.path.join(project_dir, "log4j_properties", "logs")
    # Create the directory where the log files must be kept if not present
    os.makedirs(log_dir, exist_ok=True)

    conf = get_spark_app_config()
    spark = (
        SparkSession
        .builder
        .config(conf=conf)
        .config("spark.driver.extraJavaOptions",
                f"-Dlog4j.configuration=file:{log4j_config_path} -Dcustom.log.dir={log_dir}")
        .config("spark.executor.extraJavaOptions",
                f"-Dlog4j.configuration=file:{log4j_config_path} -Dcustom.log.dir={log_dir}")
        .config("spark.jars.packages", "org.apache.spark:spark-avro_2.13:4.0.1")
        .enableHiveSupport()
        .getOrCreate()
    )

    # initialize logger class 
    logger = Log4j(spark)

    # initialize the spark dataframe logger 
    sp_df_logger = LogSparkDataframe(spark)

    # logging some debug related stuff 
    logger.debug(f"log4j.properties file dir = {log4j_config_path}")
    logger.debug(f"log files dir = {log_dir}")
    logger.debug(f"log dir exists = {os.path.exists(log_dir)}")
    
    logger.info("Reading the data from the directory")
    dataset_dir = os.path.join(project_dir,"dataset")

    """INGETING DATA FROM VARIOUS FILE FORMATS STARTS"""
    # The function must taken in file_dir csv file and then returns a spark dataFrame
    # import data from a parquet file
    ingest_data = IngestData(spark)
    # import data from a parquet file
    file_name = conf.get("file_name_parquet")
    file_dir = os.path.join(dataset_dir,file_name)
    logger.debug(f"file_name_json dir = {file_dir}")
    spark_df_parquet = ingest_data.import_data_parquet(file_dir=file_dir)
    # log spark_df_parquet dataframe
    sp_df_logger.log_df(spark_df=spark_df_parquet,spark_df_name="spark_df_parquet")
    """INGETING DATA FROM VARIOUS FILE FORMATS ENDS"""

    """Save the data in the spark dataFrame into a table STARTS"""
    # initialize the LoadSparkDFIntoTable class 
    save_df_to_table = LoadSparkDFIntoTable(spark)
    save_df_to_table.save_df_to_spark_managed_table(spark_df=spark_df_parquet,partition_config=[5,"OP_CARRIER","ORIGIN"],mode = "overwrite",db_name = conf.get("db_name"),table_name=conf.get("flight_table_name"))
    # check if the table holds the data in it
    save_df_to_table.generate_logs(conf=conf)    
    """Save the data in the spark dataFrame into a table ENDS"""

    # This line is for debugging only comment after <required to see the partitions of spark dataFrame>
    # input("Please enter")
    spark.stop()
```
#### Explaination :
- Creating a managed table in apache spark is going to need a persistant meta-store.
- Spark depends on HIVE meta-store hence we will be needing the spark-hive for this example.
- Since we want to use the HIVE meta-store here in this application I am going to enable hive support using this code ```enableHiveSupport()```
    - tells Spark to start a SparkSession with Apache Hive integration enabled
    - Allow Spark to read from, write to, and manage Hive tables and the Hive Metastore.
    - **What happens internally?**
        - Loads the Hive-specific SparkSession extensions, so that SQL features like CREATE TABLE, INSERT INTO, DESCRIBE DATABASE, etc., behave like in Hive.
        - Initializes a Hive-compatible catalog (by default named hive):
            - Metadata is stored in a Hive Metastore (a relational DB such as MySQL, Derby, or Postgres).
            - Spark SQL commands like spark.sql("SHOW TABLES") or spark.catalog.listDatabases() now use that metastore.
        - Makes Hive SerDes and file formats available
            - e.g., CREATE TABLE ... STORED AS PARQUET
            - or STORED AS AVRO (if Hive Avro SerDe is configured)
        - Allows access to Hive UDFs, UDAFs, and UDTFs
            - You can call Hive’s custom functions directly from Spark SQL.
    - **Typical use cases :**
        - Query existing Hive tables (from a Hive cluster or metastore).
        - Create new Hive-managed tables from Spark SQL.
        - Integrate Spark with a shared Hive Metastore (used by other systems).
        - Use HiveQL features not available in plain Spark SQL.
    - **Common pitfalls :**
        - Metastore conflicts
            - If you don’t specify a Hive metastore URI, Spark will spin up a local Derby metastore.
            - If two Spark sessions try to use the same Derby DB → “metastore is locked” error.
            - Fix: set a shared or distinct metastore URI:
            ```python
            .config("hive.metastore.uris", "thrift://localhost:9083")
            ```
            - Hive jars not available
                - If running Spark without built-in Hive support (some distributions), .enableHiveSupport() may throw classpath errors.
            - Performance overhead
                - It adds Hive-related initialization and dependencies even if you don’t need Hive.
-```.saveAsTable(table_name)```
    - This method takes the table_name in string type and creates the managed table in the current spark database.
    - Apache spark comes with one default database and the database name itself is default.
    - so ```.saveAsTable(table_name)``` its going to create my table in the default database.
    - Suppose you don't want to save the tables that you create in the default database and you want to create your own database where the table that you create must be stored you can do that:
        - In this case you have two options
            - pre-fix the database name with your table name ```AIRLINE_DB.table_name```
#### Error I faced :
- **Records in the table and the no of records in the table were not logged properly**
    - I wanted to log the table data and the not of rows it contains in it but all I got in the info.logs was this 
    ```bash
        INFO  pyspark-shell:244 - null
        INFO  pyspark-shell:244 - The no of records in flight_data table => DataFrame[count(1): bigint]
    ```
    - The reasons for this to happen:
        - What I was doing in my code 
        ```python
        result_df.show(25, truncate=False)
        self.logger.info(result_df.show(10, truncate=False))
        ```
        ```python
        query = f"""SELECT COUNT(*) as total FROM {table_name}"""
        self.logger.info(f"The no of records in {table_name} table => {self.spark.sql(query)}")
        ```
        - I am passing the result of DataFrame.show() (or a DataFrame object) into the logger.
        - But .show() prints to stdout (the console) and returns None
        - That’s why my log file shows:
        ```bash
            INFO  pyspark-shell:244 - null
            INFO  pyspark-shell:244 - The no of records in flight_data table => DataFrame[count(1): bigint]
        ```
        - and not the actual table data or count — because:
            - show() printed to terminal (stdout),
            - and then returned None, which is what got logged.
#### solution
- I need to explicitly capture the data as strings, not rely on .show() to print.
- Here is the corrected code 
```python
def generate_logs(self, conf):
    try:
        db_name = conf.get("db_name")
        table_name = conf.get("flight_table_name")

        # Switch to the correct database
        if db_name:
            self.spark.catalog.setCurrentDatabase(db_name)

        # Get the first 24 records
        query = f"SELECT * FROM {table_name} LIMIT 24"
        result_df = self.spark.sql(query)

        # Capture the DataFrame as a formatted string (no stdout print)
        result_str = result_df._jdf.showString(24, 0, False)
        self.logger.info(f"\nFirst 24 records from table '{table_name}':\n{result_str}")

        # Get count value properly
        count_df = self.spark.sql(f"SELECT COUNT(*) as total FROM {table_name}")
        count_value = count_df.collect()[0]['total']
        self.logger.info(f"The number of records in table '{table_name}' => {count_value}")

        # Log schema
        self.logger.debug(f"Schema of table '{db_name}.{table_name}':")
        for field in result_df.schema.fields:
            self.logger.debug(f"  {field.name}: {field.dataType.simpleString()}")

    except Exception as e:
        self.logger.error(str(e))
        raise
```
- ```result_str = result_df._jdf.showString(24, 0, False)```
    - It is the internal JVM method Spark uses to format the show() output as a string, without printing to stdout.
    - That’s why this will appear properly in the info.log.
- **Why the count wasn’t logged either?**
    - ```self.logger.info(f"The no of records in {table_name} table => {self.spark.sql(query)}")```
        - logs the DataFrame object reference, not the query result.
        - self.spark.sql(query) returns a DataFrame, not a number.
        - So I must collect the value like this :
        ```python
        count_value = self.spark.sql(f"SELECT COUNT(*) as total FROM {table_name}").collect()[0]['total']
        self.logger.info(f"The number of records in table '{table_name}' => {count_value}")
        ```
